# LoRA vs QLoRA 비교 학습 (Colab)

`training/tagging/train_lora.py` 를 **같은 인자로 `--load-in-4bit` 만 바꿔** 두 번 돌리고, 각 실행이 남긴
`run_metrics.json`(최대 GPU 메모리·학습 시간·비용·eval loss)을 표로 모은다. 학습 코드는 이 노트북에 없다 — 저장소 스크립트 그대로다.

| 순서 | 셀 | 하는 일 |
|---|---|---|
| 1 | GPU 확인 | **L4·A100 권장.** T4 는 BF16 이 없어 LoRA(16비트 원본)가 FP32 로 올라가 메모리가 넘친다 |
| 2 | ⚙️ 설정 | `TASK`, 데이터 위치, GPU 단가. **여기만 바꾼다** |
| 3 | 준비 | 드라이브 연결, 저장소 클론, `training/requirements.txt` 설치 |
| 4 | LoRA 학습 | `train_lora.py` |
| 5 | QLoRA 학습 | `train_lora.py --load-in-4bit` (나머지 인자는 4번과 같음) |
| 6 | 비교표 | 메모리·시간·비용·eval loss |
| 7 | 다음 단계 | vLLM 서빙과 품질 평가 |

**학습 데이터는 맥에서 만들어 드라이브에 올린다** (`build_sft.py` 가 서빙 코드의 프롬프트를 import 하므로 `uv sync` 된 저장소에서 만든다).

```bash
# 태깅
uv run python training/tagging/from_teacher.py data/teacher/pilot200/tagging.jsonl --out data/tagging.jsonl
uv run python training/tagging/build_sft.py data/tagging.jsonl --out training/tagging/out --allow-unreviewed
# 추출
uv run python training/extraction/from_teacher.py data/teacher/pilot200/extraction.jsonl --out training/extraction/out
```
→ `training/tagging/out/`, `training/extraction/out/` 폴더(train.jsonl · val.jsonl · manifest.json)를 드라이브 `ktb-ai/sft/` 아래에 그대로 올린다.

> `--allow-unreviewed` 는 사람 검수 전 teacher 라벨로 **파이프라인을 시험할 때만** 쓴다. 이 노트북의 결과는 LoRA·QLoRA 비용 비교용이다.

## 1. GPU 확인

In [ ]:
import subprocess

import torch

print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True
    ).stdout
)
if not torch.cuda.is_bf16_supported():
    print(
        "⚠️ BF16 미지원 GPU(T4 등). train_lora.py 는 이때 원본을 FP32 로 올리므로 LoRA 비교가 메모리 부족으로 실패할 수 있다."
    )
    print("   런타임 → 런타임 유형 변경에서 L4 또는 A100 을 고른다.")

## 2. ⚙️ 설정 — 이 셀만 바꾼다

| 값 | 뜻 |
|---|---|
| `TASK` | `"tagging"` 또는 `"extraction"` |
| `GPU_HOURLY_USD` | GPU 시간당 단가(달러). 비용 칸 계산용. 모르면 0 |
| `COMMON_ARGS` | 두 모드에 **똑같이** 들어가는 학습 인자. 비교 중에는 바꾸지 않는다 |

길이(`--max-length`)는 pilot200 최대 길이(태깅 975, 추출 3,257 토큰)에 여유를 둔 값이다.
두 모드 모두 gradient checkpointing 을 켜서 조건을 맞춘다.

In [ ]:
TASK = "tagging"  # "tagging" 또는 "extraction"
BRANCH = "feat/lena-lora-qlora-compare"
BASE_MODEL = "kakaocorp/kanana-2-3b-instruct"
SFT_ROOT = "/content/drive/MyDrive/ktb-ai/sft"  # 그 아래 tagging/out, extraction/out
OUT_ROOT = "/content/drive/MyDrive/ktb-ai/adapters"
GPU_HOURLY_USD = 0.0  # 예: L4 약 0.7, A100 약 1.2 (클라우드 시세 기준. Colab 은 컴퓨팅 단위 환산)
SEED = 42

DATA_DIR = f"{SFT_ROOT}/{TASK}/out"
COMMON_ARGS = {
    "tagging": "--max-length 1280 --batch-size 8 --grad-accum 2",
    "extraction": "--max-length 4096 --batch-size 1 --grad-accum 16",
}[TASK] + f" --epochs 3 --learning-rate 2e-4 --lora-r 16 --lora-alpha 32 --seed {SEED} --gradient-checkpointing"
print(TASK, DATA_DIR)
print(COMMON_ARGS)

## 3. 준비 — 드라이브 연결, 저장소 클론, 학습 의존성 설치

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
for name in ("train.jsonl", "val.jsonl", "manifest.json"):
    path = f"{DATA_DIR}/{name}"
    print(("✅ " if os.path.exists(path) else "❌ 없음: ") + path)

!rm -rf /content/repo && git clone -q --depth 1 -b {BRANCH} https://github.com/100-hours-a-week/KTB4-17th-AI /content/repo
%cd /content/repo
!pip install -q -r training/requirements.txt

## 4. LoRA 학습 (원본 16비트)
`!python` 으로 따로 실행하므로 끝나면 GPU 메모리가 비워진다. 런타임을 다시 시작하지 않아도 된다.

In [ ]:
RUN = f"{OUT_ROOT}/{TASK}-lora-s{SEED}"
!python training/tagging/train_lora.py --base-model {BASE_MODEL} --data-dir {DATA_DIR} --output-dir {RUN} {COMMON_ARGS} --gpu-hourly-usd {GPU_HOURLY_USD}

## 5. QLoRA 학습 (원본 4비트 NF4) — 4번과 인자가 같고 `--load-in-4bit` 만 다르다

In [ ]:
RUN = f"{OUT_ROOT}/{TASK}-qlora-s{SEED}"
!python training/tagging/train_lora.py --base-model {BASE_MODEL} --data-dir {DATA_DIR} --output-dir {RUN} {COMMON_ARGS} --gpu-hourly-usd {GPU_HOURLY_USD} --load-in-4bit

## 6. 비교표

| 칸 | 뜻 |
|---|---|
| `train_peak_gib` | 학습 중 실제로 쓴 최대 GPU 메모리 |
| `train_reserved_gib` | PyTorch 가 잡아 둔 메모리 (`nvidia-smi` 값에 가까움) |
| `train_seconds`, `train_cost_usd` | 학습 시간, 시간 × `GPU_HOURLY_USD` |
| `eval_loss` | 검증 손실. 품질 판정은 7번의 평가 점수로 한다 |

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
for mode in ("lora", "qlora"):
    path = Path(OUT_ROOT) / f"{TASK}-{mode}-s{SEED}" / "run_metrics.json"
    if path.exists():
        rows.append(json.loads(path.read_text()))
    else:
        print("아직 없음:", path)
columns = [
    "mode",
    "gpu",
    "dtype",
    "train_peak_gib",
    "train_reserved_gib",
    "train_seconds",
    "train_cost_usd",
    "eval_loss",
    "adapter_mb",
]
display(pd.DataFrame(rows)[columns] if rows else "기록 없음")

## 7. 다음 단계 — 품질 점수

비용만으로는 고르지 않는다. 두 어댑터를 **같은 서빙 조건**(같은 원본 정밀도)에 올려 품질을 잰다.

1. 어댑터(`{OUT_ROOT}/{TASK}-lora-s42/adapter`, `...-qlora-s42/adapter`)를 vLLM 에 함께 올린다 — `serving/README.md`, `serving/vllm/config.yaml`
2. 태깅: `uv run python scripts/eval_tagging.py --slm --model <어댑터 이름>` (regression split)
3. Langfuse 실험으로 남기려면: `LLM_MODEL=<어댑터 이름> uv run python scripts/run_langfuse_calibration.py --dataset quality/persona/onboarding-tagging --split regression`
   - 실행 단위 점수 `tagging_primary_f1`, `tagging_exact_match`, 추출은 `build_dimension_judgment_accuracy` 가 같이 기록된다
   - 앱은 `LLM_MODEL` 하나로 모든 기능을 부르므로 `--dataset` 을 어댑터 작업과 맞춰 하나씩 돌린다

판정(제안): 품질 차이가 1%p 이내면 더 싼 쪽, 넘으면 품질이 좋은 쪽을 고르되 그 차이가 비용 차이만큼 가치 있는지 표로 남긴다.